In [1]:
import numpy as np
import pandas as pd

In [31]:
expenses_csv = """expense_id,user_id,category_id,payment_mode,amount,expense_date
1,1,1,UPI,$250,2026-05-01
2,1,2,cash,$12000,2026-05-02
3,1,4,card,$1500,2026-05-05
4,2,1,card,$300,2026-05-01
5,2,6,UPI,$2000,2026-05-03
6,1,3,UPI,₹400,2026/06/01
7,1,5,cash,700,06-06-2026
8,2,3,card,$999,2026-05-01
"""

with open("expenses.csv", "w") as f:
  f.write(expenses_csv)

df = pd.read_csv("expenses.csv")
print(df)

   expense_id  user_id  category_id payment_mode  amount expense_date
0           1        1            1          UPI    $250   2026-05-01
1           2        1            2         cash  $12000   2026-05-02
2           3        1            4         card   $1500   2026-05-05
3           4        2            1         card    $300   2026-05-01
4           5        2            6          UPI   $2000   2026-05-03
5           6        1            3          UPI    ₹400   2026/06/01
6           7        1            5         cash     700   06-06-2026
7           8        2            3         card    $999   2026-05-01


In [32]:
df["amount"] = (
    df["amount"].astype(str).replace(r"[₹$,]","", regex=True).astype(float)
)
print(df)

   expense_id  user_id  category_id payment_mode   amount expense_date
0           1        1            1          UPI    250.0   2026-05-01
1           2        1            2         cash  12000.0   2026-05-02
2           3        1            4         card   1500.0   2026-05-05
3           4        2            1         card    300.0   2026-05-01
4           5        2            6          UPI   2000.0   2026-05-03
5           6        1            3          UPI    400.0   2026/06/01
6           7        1            5         cash    700.0   06-06-2026
7           8        2            3         card    999.0   2026-05-01


In [34]:
df["expense_date"] = df["expense_date"].astype(str).str.strip()

df["expense_date"] = pd.to_datetime(df["expense_date"], errors="coerce", dayfirst=True)
df = df.dropna(subset=["expense_date"])
print(df)

   expense_id  user_id  category_id payment_mode   amount expense_date
0           1        1            1          UPI    250.0   2026-01-05
1           2        1            2         cash  12000.0   2026-02-05
2           3        1            4         card   1500.0   2026-05-05
3           4        2            1         card    300.0   2026-01-05
4           5        2            6          UPI   2000.0   2026-03-05
7           8        2            3         card    999.0   2026-01-05


In [42]:
df["Month"] = df["expense_date"].dt.to_period("M")
print(df)

   expense_id  user_id  category_id payment_mode   amount expense_date  \
0           1        1            1          UPI    250.0   2026-01-05   
1           2        1            2         cash  12000.0   2026-02-05   
2           3        1            4         card   1500.0   2026-05-05   
3           4        2            1         card    300.0   2026-01-05   
4           5        2            6          UPI   2000.0   2026-03-05   
7           8        2            3         card    999.0   2026-01-05   

     month    Month  
0  2026-01  2026-01  
1  2026-02  2026-02  
2  2026-05  2026-05  
3  2026-01  2026-01  
4  2026-03  2026-03  
7  2026-01  2026-01  


In [43]:
monthly_total = df.groupby("Month")["amount"].sum()
monthly_avg = df.groupby("Month")["amount"].mean()

monthly_total_arr = np.array(monthly_total)
monthly_avg_arr = np.array(monthly_avg)

print("Monthly total: ", monthly_total_arr)
print("Monthly Averages: ", monthly_avg_arr)

Monthly total:  [ 1549. 12000.  2000.  1500.]
Monthly Averages:  [  516.33333333 12000.          2000.          1500.        ]


In [44]:
#category_wise_breakdown
cat_breakdown = (
    df.groupby(["Month", "category_id"])["amount"].sum().unstack().fillna(0)
)
print(cat_breakdown)

category_id      1        2      3       4       6
Month                                             
2026-01      550.0      0.0  999.0     0.0     0.0
2026-02        0.0  12000.0    0.0     0.0     0.0
2026-03        0.0      0.0    0.0     0.0  2000.0
2026-05        0.0      0.0    0.0  1500.0     0.0
